<a href="https://colab.research.google.com/github/logonia/DAP/blob/main/Improved_Ablation_CapFG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# final_complete_capfg.py
# - Working ablation (Baseline, MaskOnly, CapFGOnly, Full)
# - ResNet-18+CBAM benchmark
# - Confusion matrix + training plots
# - Threshold sensitivity & background shift
# - Complexity, IoU, leakage
# ============================================================

import os
import random
import time
import argparse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam, lr_scheduler
from torch.utils.data import DataLoader, random_split, Subset, Dataset
from torchvision import datasets, transforms

# ---------------------------
# Reproducibility
# ---------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ---------------------------
# Helper functions
# ---------------------------
def squash(inputs, axis=-1):
    norm = torch.norm(inputs, p=2, dim=axis, keepdim=True)
    scale = (norm ** 2) / (1.0 + norm ** 2)
    return scale * inputs / (norm + 1e-8)

def to_onehot(y, num_classes):
    return torch.eye(num_classes, device=y.device)[y]

def caps_loss(y_true, y_pred, x, x_recon, lam_recon):
    margin = (y_true * torch.clamp(0.9 - y_pred, min=0.0)**2 +
              0.5 * (1.0 - y_true) * torch.clamp(y_pred - 0.1, min=0.0)**2)
    margin_loss = margin.sum(dim=1).mean()
    recon_loss = F.mse_loss(x_recon, x)
    return margin_loss + lam_recon * recon_loss

def mean_ci(values, confidence=0.95):
    arr = np.array(values)
    n = len(arr)
    mean = np.mean(arr)
    std = np.std(arr, ddof=1) if n > 1 else 0.0
    if n > 1:
        h = stats.t.ppf((1+confidence)/2.0, n-1) * (std / np.sqrt(n))
    else:
        h = 0.0
    return mean, std, mean-h, mean+h

# ---------------------------
# Data loaders (CIFAR-10) with optional threshold mask
# ---------------------------
class ThresholdMaskTransform:
    def __init__(self, threshold):
        self.threshold = threshold
    def __call__(self, x):
        return x * (x > self.threshold).float()

class BackgroundNoiseTransform:
    def __init__(self, noise_std=0.25):
        self.noise_std = noise_std
    def __call__(self, x):
        return torch.clamp(x + torch.randn_like(x) * self.noise_std, 0.0, 1.0)

def load_cifar10(data_dir="./data", batch_size=128, val_size=5000, threshold=None):
    train_ops = [transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(), transforms.ToTensor()]
    eval_ops = [transforms.ToTensor()]
    if threshold is not None:
        train_ops.append(ThresholdMaskTransform(threshold))
        eval_ops.append(ThresholdMaskTransform(threshold))
    train_tf = transforms.Compose(train_ops)
    eval_tf = transforms.Compose(eval_ops)

    full_train_aug = datasets.CIFAR10(data_dir, train=True, download=True, transform=train_tf)
    full_train_eval = datasets.CIFAR10(data_dir, train=True, download=False, transform=eval_tf)
    test_set = datasets.CIFAR10(data_dir, train=False, download=True, transform=eval_tf)

    total_len = len(full_train_aug)
    train_len = total_len - val_size
    train_subset, val_subset = random_split(range(total_len), [train_len, val_size],
                                            generator=torch.Generator().manual_seed(42))
    train_set = Subset(full_train_aug, train_subset.indices)
    val_set = Subset(full_train_eval, val_subset.indices)

    pin = torch.cuda.is_available()
    train_loader = DataLoader(train_set, batch_size, shuffle=True, num_workers=2, pin_memory=pin)
    val_loader   = DataLoader(val_set, batch_size, shuffle=False, num_workers=2, pin_memory=pin)
    test_loader  = DataLoader(test_set, batch_size, shuffle=False, num_workers=2, pin_memory=pin)
    return train_loader, val_loader, test_loader

def load_cifar10_corrupted_test(data_dir="./data", batch_size=128, noise_std=0.25):
    transform = transforms.Compose([transforms.ToTensor(), BackgroundNoiseTransform(noise_std)])
    test_set = datasets.CIFAR10(data_dir, train=False, download=True, transform=transform)
    return DataLoader(test_set, batch_size, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())

# ---------------------------
# Synthetic dataset for background suppression
# ---------------------------
class SyntheticForegroundDataset(Dataset):
    def __init__(self, num_samples=2000, size=32, num_classes=10):
        self.num_samples = num_samples
        self.size = size
        self.num_classes = num_classes
        self.images = []
        self.labels = []
        self.masks = []
        for _ in range(num_samples):
            img = torch.randn(3, size, size) * 0.5
            label = random.randint(0, num_classes-1)
            fg_size = random.randint(8, 16)
            fx = random.randint(0, size - fg_size)
            fy = random.randint(0, size - fg_size)
            mask = torch.zeros(1, size, size)
            mask[:, fy:fy+fg_size, fx:fx+fg_size] = 1.0
            fg = torch.randn(3, fg_size, fg_size) * 0.2 + 0.8
            img[:, fy:fy+fg_size, fx:fx+fg_size] = fg
            self.images.append(img)
            self.labels.append(label)
            self.masks.append(mask)
    def __len__(self):
        return self.num_samples
    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx], self.masks[idx]

# ---------------------------
# Capsule components (working version)
# ---------------------------
class PrimaryCapsuleBase(nn.Module):
    def __init__(self, in_c=256, maps=32, dims=8):
        super().__init__()
        self.maps, self.dims = maps, dims
        self.conv = nn.Conv2d(in_c, maps*dims, 9, stride=2)
    def forward(self, x):
        out = self.conv(x)
        b, _, h, w = out.shape
        out = out.view(b, self.maps, self.dims, h, w).permute(0,1,3,4,2).contiguous()
        out = squash(out, axis=-1)
        return out.reshape(b, -1, self.dims), h, w

class PrimaryCapsuleUnsquashed(nn.Module):
    def __init__(self, in_c=256, maps=32, dims=8):
        super().__init__()
        self.maps, self.dims = maps, dims
        self.conv = nn.Conv2d(in_c, maps*dims, 9, stride=2)
    def forward(self, x):
        out = self.conv(x)
        b, _, h, w = out.shape
        out = out.view(b, self.maps, self.dims, h, w).permute(0,1,3,4,2).contiguous()
        raw_norms = torch.norm(out, dim=-1)
        return out, raw_norms, h, w

class DenseCapsule(nn.Module):
    def __init__(self, in_caps, out_caps, in_dims, out_dims, routings=3):
        super().__init__()
        self.in_caps = in_caps
        self.out_caps = out_caps
        self.routings = routings
        self.W = nn.Parameter(0.01 * torch.randn(out_caps, in_caps, out_dims, in_dims))
    def forward(self, x):
        u_hat = torch.einsum('bid,oijd->boij', x, self.W)
        b = torch.zeros(x.size(0), self.out_caps, self.in_caps, device=x.device)
        for i in range(self.routings):
            c = F.softmax(b, dim=1)
            s = (c.unsqueeze(-1) * u_hat).sum(dim=2)
            v = squash(s, axis=-1)
            if i < self.routings-1:
                b = b + (u_hat * v.unsqueeze(2)).sum(dim=-1)
        return v

class ImprovedCapFG(nn.Module):
    def __init__(self, in_maps=32, beta=0.2, entropy_weight=0.01, use_global=True):
        super().__init__()
        self.beta = beta
        self.entropy_weight = entropy_weight
        self.use_global = use_global
        self.local = nn.Sequential(
            nn.Conv2d(in_maps, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, 1)
        )
        if use_global:
            self.global_pool = nn.AdaptiveAvgPool2d(1)
            self.fc = nn.Linear(in_maps, 1)
        self.temperature = nn.Parameter(torch.tensor(1.0))
    def forward(self, raw_norms):
        logits = self.local(raw_norms)
        if self.use_global:
            g = torch.sigmoid(self.fc(self.global_pool(raw_norms).flatten(1)))
            logits = logits * g[:, :, None, None]
        return torch.sigmoid(logits / self.temperature)
    def entropy_loss(self, mask):
        m = mask.view(mask.size(0), -1)
        entropy = -torch.mean(m * torch.log(m+1e-8) + (1-m)*torch.log(1-m+1e-8))
        return self.entropy_weight * entropy

class CapsuleNetAblation(nn.Module):
    def __init__(self, shape=(3,32,32), classes=10, routings=3,
                 use_input_mask=False, use_capfg=False, mask_threshold=0.1,
                 beta=0.2, entropy_weight=0.01, use_global=True):
        super().__init__()
        self.shape = shape
        self.classes = classes
        self.use_input_mask = use_input_mask
        self.use_capfg = use_capfg
        self.mask_threshold = mask_threshold

        self.conv1 = nn.Conv2d(shape[0], 256, kernel_size=9, stride=1, padding=0)
        self.relu = nn.ReLU(inplace=True)

        if use_capfg:
            self.primary = PrimaryCapsuleUnsquashed(256, 32, 8)
            self.mask_gen = ImprovedCapFG(32, beta=beta, entropy_weight=entropy_weight, use_global=use_global)
            self.num_caps_in = 32 * 8 * 8
        else:
            self.primary = PrimaryCapsuleBase(256, 32, 8)
            self.num_caps_in = 32 * 8 * 8

        self.digitcaps = DenseCapsule(self.num_caps_in, classes, 8, 16, routings)
        self.decoder = nn.Sequential(
            nn.Linear(16*classes, 512), nn.ReLU(inplace=True),
            nn.Linear(512, 1024), nn.ReLU(inplace=True),
            nn.Linear(1024, shape[0]*shape[1]*shape[2]), nn.Sigmoid()
        )

    def forward(self, x, y=None, return_mask=False):
        if self.use_input_mask:
            x = x * (x > self.mask_threshold).float()
        out = self.relu(self.conv1(x))
        if self.use_capfg:
            prim_unsq, raw_norms, h, w = self.primary(out)
            mask = self.mask_gen(raw_norms)
            mask_exp = mask.unsqueeze(-1)
            prim_masked = prim_unsq * mask_exp + prim_unsq * self.mask_gen.beta * (1 - mask_exp)
            prim_squashed = squash(prim_masked, axis=-1)
            prim_flat = prim_squashed.view(x.size(0), -1, 8)
        else:
            prim_flat, h, w = self.primary(out)
        out = self.digitcaps(prim_flat)
        length = out.norm(dim=-1)
        if y is None:
            idx = length.max(1)[1]
            y = torch.eye(self.classes, device=x.device)[idx]
        recon = self.decoder((out * y[:,:,None]).view(out.size(0), -1))
        recon = recon.view(-1, *self.shape)
        if return_mask and self.use_capfg:
            return length, recon, mask
        return length, recon

    def mask_entropy_loss(self, mask):
        if self.use_capfg:
            return self.mask_gen.entropy_loss(mask)
        return torch.tensor(0.0, device=device)

# ---------------------------
# ResNet-18 + CBAM (benchmark)
# ---------------------------
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels//reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels//reduction, channels, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg = self.fc(self.avg_pool(x).view(x.size(0), -1)).view(x.size(0), x.size(1), 1, 1)
        max_ = self.fc(self.max_pool(x).view(x.size(0), -1)).view(x.size(0), x.size(1), 1, 1)
        return self.sigmoid(avg + max_)

class SpatialAttention(nn.Module):
    def __init__(self, kernel=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel, padding=kernel//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        max_, _ = torch.max(x, dim=1, keepdim=True)
        concat = torch.cat([avg, max_], dim=1)
        return self.sigmoid(self.conv(concat))

class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1, use_cbam=True):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, 1, stride, bias=False),
                nn.BatchNorm2d(planes)
            )
        self.cbam = CBAMBlock(planes) if use_cbam else nn.Identity()
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.cbam(out)
        out += self.shortcut(x)
        return F.relu(out)

class CBAMBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.channel = ChannelAttention(channels)
        self.spatial = SpatialAttention()
    def forward(self, x):
        x = x * self.channel(x)
        x = x * self.spatial(x)
        return x

class ResNetCBAM(nn.Module):
    def __init__(self, num_classes=10, use_cbam=True):
        super().__init__()
        self.in_planes = 64
        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(64, 2, stride=1, use_cbam=use_cbam)
        self.layer2 = self._make_layer(128, 2, stride=2, use_cbam=use_cbam)
        self.layer3 = self._make_layer(256, 2, stride=2, use_cbam=use_cbam)
        self.layer4 = self._make_layer(512, 2, stride=2, use_cbam=use_cbam)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.linear = nn.Linear(512, num_classes)

    def _make_layer(self, planes, num_blocks, stride, use_cbam):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(BasicBlock(self.in_planes, planes, stride, use_cbam))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        return self.linear(out)

def resnet18_cbam(num_classes=10):
    return ResNetCBAM(num_classes, use_cbam=True)

# ---------------------------
# Training and evaluation for capsule models
# ---------------------------
def evaluate_capsule(model, loader, lam_recon, num_classes, use_mask_loss=False):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            y_onehot = to_onehot(y, num_classes)
            if use_mask_loss and hasattr(model, 'mask_entropy_loss'):
                y_pred, x_recon, mask = model(x, return_mask=True)
                loss = caps_loss(y_onehot, y_pred, x, x_recon, lam_recon) + model.mask_entropy_loss(mask)
            else:
                y_pred, x_recon = model(x)
                loss = caps_loss(y_onehot, y_pred, x, x_recon, lam_recon)
            total_loss += loss.item() * x.size(0)
            correct += (y_pred.argmax(1) == y).sum().item()
            total += x.size(0)
    return total_loss/total, correct/total

def train_capsule_one_run(model, train_loader, val_loader, config, use_mask_loss=False, history=None):
    opt = Adam(model.parameters(), lr=config['lr'])
    sched = lr_scheduler.ExponentialLR(opt, gamma=config['lr_decay'])
    best_val_acc = 0.0
    train_losses, val_accs = [], []
    for epoch in range(config['epochs']):
        model.train()
        epoch_loss = 0.0
        total = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            y_onehot = to_onehot(y, config['classes'])
            opt.zero_grad()
            if use_mask_loss and hasattr(model, 'mask_entropy_loss'):
                y_pred, x_recon, mask = model(x, y_onehot, return_mask=True)
                loss = caps_loss(y_onehot, y_pred, x, x_recon, config['lam_recon']) + model.mask_entropy_loss(mask)
            else:
                y_pred, x_recon = model(x, y_onehot)
                loss = caps_loss(y_onehot, y_pred, x, x_recon, config['lam_recon'])
            loss.backward()
            opt.step()
            epoch_loss += loss.item() * x.size(0)
            total += x.size(0)
        sched.step()
        _, val_acc = evaluate_capsule(model, val_loader, config['lam_recon'], config['classes'], use_mask_loss)
        train_losses.append(epoch_loss/total)
        val_accs.append(val_acc)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
        print(f"  Epoch {epoch+1:02d}/{config['epochs']} | loss={epoch_loss/total:.4f} | val_acc={val_acc:.4f} | best={best_val_acc:.4f}")
    if history is not None:
        history['train_loss'] = train_losses
        history['val_acc'] = val_accs
    return best_val_acc

def multi_run_capsule(experiments, config, train_loader, val_loader, n_runs=3):
    results = {}
    for exp in experiments:
        name = exp['name']
        print(f"\n--- {name} ---")
        accs = []
        for run in range(n_runs):
            set_seed(42+run)
            model = CapsuleNetAblation(**exp['model_kwargs']).to(device)
            acc = train_capsule_one_run(model, train_loader, val_loader, config, use_mask_loss=exp['use_mask_loss'])
            accs.append(acc)
            print(f"  Run {run+1} final val_acc = {acc:.4f}")
        mean, std, lo, hi = mean_ci(accs)
        results[name] = (mean, std, lo, hi, accs)
        print(f"{name}: {mean:.4f} ± {std:.4f} | 95% CI [{lo:.4f}, {hi:.4f}]")
    return results

# ---------------------------
# ResNet training
# ---------------------------
def train_resnet_one_run(model, train_loader, val_loader, epochs, lr=0.1):
    criterion = nn.CrossEntropyLoss()
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_acc = 0.0
    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            opt.step()
        scheduler.step()
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                out = model(x)
                _, pred = out.max(1)
                correct += (pred == y).sum().item()
                total += y.size(0)
        val_acc = correct / total
        if val_acc > best_acc:
            best_acc = val_acc
        print(f"Epoch {epoch+1}/{epochs} | val_acc={val_acc:.4f} | best={best_acc:.4f}")
    return best_acc

def multi_run_resnet(model_class, model_kwargs, config, train_loader, val_loader, n_runs=3):
    accs = []
    for run in range(n_runs):
        set_seed(42+run)
        model = model_class(**model_kwargs).to(device)
        acc = train_resnet_one_run(model, train_loader, val_loader, config['epochs'])
        accs.append(acc)
        print(f"ResNet run {run+1} best val_acc = {acc:.4f}")
    return mean_ci(accs)

# ---------------------------
# Background suppression, complexity
# ---------------------------
def background_suppression_metrics(model, synth_loader):
    model.eval()
    ious, leakages = [], []
    with torch.no_grad():
        for x, y, true_mask in synth_loader:
            x = x.to(device)
            true_mask = true_mask.to(device)
            _, _, pred_mask = model(x, return_mask=True)
            pred_up = F.interpolate(pred_mask, size=true_mask.shape[-2:], mode='bilinear', align_corners=False)
            pred_bin = (pred_up > 0.5).float()
            inter = (pred_bin * true_mask).sum(dim=(1,2,3))
            union = ((pred_bin + true_mask) > 0.5).float().sum(dim=(1,2,3))
            iou = (inter+1e-6)/(union+1e-6)
            ious.extend(iou.cpu().numpy())
            bg_leak = (pred_bin * (1-true_mask)).sum(dim=(1,2,3)) / ((1-true_mask).sum(dim=(1,2,3))+1e-6)
            leakages.extend(bg_leak.cpu().numpy())
    return np.mean(ious), np.mean(leakages)

def measure_complexity(model, input_shape, is_capsule=True, use_mask_loss=False):
    model.eval()
    dummy = torch.randn(1, *input_shape).to(device)
    params = sum(p.numel() for p in model.parameters())
    with torch.no_grad():
        for _ in range(10):
            if is_capsule:
                if use_mask_loss and hasattr(model, 'mask_entropy_loss'):
                    model(dummy, return_mask=True)
                else:
                    model(dummy)
            else:
                model(dummy)
    start = time.time()
    with torch.no_grad():
        for _ in range(100):
            if is_capsule:
                if use_mask_loss and hasattr(model, 'mask_entropy_loss'):
                    model(dummy, return_mask=True)
                else:
                    model(dummy)
            else:
                model(dummy)
    infer_ms = (time.time() - start) / 100 * 1000
    return params, infer_ms

def run_threshold_sensitivity(config, thresholds=(0.05,0.10,0.15,0.20), n_runs=3):
    rows = []
    for th in thresholds:
        print(f"\nThreshold {th}")
        train_loader, val_loader, _ = load_cifar10(batch_size=config['batch_size'], threshold=th)
        # MaskOnly
        mask_accs = []
        for run in range(n_runs):
            set_seed(100+run)
            model = CapsuleNetAblation(shape=(3,32,32), classes=10,
                                       use_input_mask=True, use_capfg=False, mask_threshold=th).to(device)
            acc = train_capsule_one_run(model, train_loader, val_loader, config, use_mask_loss=False)
            mask_accs.append(acc)
        # Full (Mask+CapFG)
        full_accs = []
        for run in range(n_runs):
            set_seed(200+run)
            model = CapsuleNetAblation(shape=(3,32,32), classes=10,
                                       use_input_mask=True, use_capfg=True, mask_threshold=th,
                                       beta=0.2, entropy_weight=0.01, use_global=True).to(device)
            acc = train_capsule_one_run(model, train_loader, val_loader, config, use_mask_loss=True)
            full_accs.append(acc)
        rows.append({
            'threshold': th,
            'mask_only_mean': np.mean(mask_accs),
            'mask_only_std': np.std(mask_accs),
            'full_mean': np.mean(full_accs),
            'full_std': np.std(full_accs)
        })
    df = pd.DataFrame(rows)
    df.to_csv('threshold_sensitivity_results.csv', index=False)
    return df

def evaluate_background_shift(model, batch_size=128, noise_levels=(0.1,0.2,0.3)):
    rows = []
    for nl in noise_levels:
        loader = load_cifar10_corrupted_test(batch_size=batch_size, noise_std=nl)
        _, acc = evaluate_capsule(model, loader, 0.0005*3*32*32, 10, use_mask_loss=True)
        rows.append({'noise_std': nl, 'test_acc': acc})
    df = pd.DataFrame(rows)
    df.to_csv('background_shift_results.csv', index=False)
    return df

def plot_training_curves(histories, save_path='training_curves.png'):
    fig, (ax1, ax2) = plt.subplots(1,2, figsize=(12,4))
    for name, hist in histories.items():
        if 'train_loss' in hist:
            ax1.plot(hist['train_loss'], label=name)
        if 'val_acc' in hist:
            ax2.plot(hist['val_acc'], label=name)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Training Loss')
    ax1.legend()
    ax1.grid(True)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Validation Accuracy')
    ax2.legend()
    ax2.grid(True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"Saved training curves to {save_path}")

def plot_ablation(results, save_path='ablation_comparison.png'):
    names = list(results.keys())
    means = [results[n][0] for n in names]
    stds  = [results[n][1] for n in names]
    plt.figure(figsize=(8,5))
    plt.bar(names, means, yerr=stds, capsize=5, color=['gray','blue','green','red'])
    plt.ylabel('Validation Accuracy')
    plt.title('Ablation Study')
    plt.ylim(0.65, 0.75)
    for i, (m, s) in enumerate(zip(means, stds)):
        plt.text(i, m+0.003, f'{m:.3f}', ha='center')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"Saved ablation plot to {save_path}")

def compute_confusion_matrix(model, loader, config, is_capsule=True, use_mask_loss=False):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if is_capsule:
                if use_mask_loss and hasattr(model, 'mask_entropy_loss'):
                    y_pred, _, _ = model(x, return_mask=True)
                else:
                    y_pred, _ = model(x)
                preds = y_pred.argmax(1)
            else:
                out = model(x)
                preds = out.argmax(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    cm = confusion_matrix(all_labels, all_preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=list(range(10)))
    disp.plot(cmap='Blues')
    plt.title('Confusion Matrix - Full CapFG Model')
    plt.savefig('confusion_matrix.png', dpi=150)
    plt.close()
    print("Saved confusion_matrix.png")
    return cm

# ---------------------------
# Main
# ---------------------------
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--n_runs', type=int, default=3, help='Number of runs for each experiment')
    parser.add_argument('--epochs', type=int, default=30, help='Number of epochs')
    parser.add_argument('--batch_size', type=int, default=128)
    args, unknown = parser.parse_known_args()
    if unknown:
        print(f"Ignoring unknown arguments: {unknown}")

    config = {
        'epochs': args.epochs,
        'batch_size': args.batch_size,
        'lr': 0.001,
        'lr_decay': 0.9,
        'lam_recon': 0.0005 * (3*32*32),
        'classes': 10,
    }

    print("\nLoading CIFAR-10 (no input mask for main ablation)...")
    train_loader, val_loader, test_loader = load_cifar10(batch_size=config['batch_size'], threshold=None)

    # Define ablative configurations
    experiments = [
        {'name': 'Baseline', 'model_kwargs': {'use_input_mask': False, 'use_capfg': False}, 'use_mask_loss': False},
        {'name': 'MaskOnly', 'model_kwargs': {'use_input_mask': True, 'use_capfg': False, 'mask_threshold': 0.1}, 'use_mask_loss': False},
        {'name': 'CapFGOnly', 'model_kwargs': {'use_input_mask': False, 'use_capfg': True,
                                               'beta':0.2, 'entropy_weight':0.01, 'use_global':True}, 'use_mask_loss': True},
        {'name': 'Full (Mask+CapFG)', 'model_kwargs': {'use_input_mask': True, 'use_capfg': True,
                                                       'mask_threshold':0.1, 'beta':0.2, 'entropy_weight':0.01, 'use_global':True}, 'use_mask_loss': True}
    ]

    print("\n" + "="*80)
    print(f"ABLATION STUDY (n_runs={args.n_runs}, epochs={args.epochs})")
    print("="*80)

    results = multi_run_capsule(experiments, config, train_loader, val_loader, n_runs=args.n_runs)

    # Train final Full model for additional metrics and record history
    print("\nTraining final Full model for detailed analysis...")
    final_model = CapsuleNetAblation(**experiments[3]['model_kwargs']).to(device)
    history = {}
    best_acc = train_capsule_one_run(final_model, train_loader, val_loader, config, use_mask_loss=True, history=history)
    print(f"Final Full model best val_acc = {best_acc:.4f}")

    # Confusion matrix on test set
    print("\nGenerating confusion matrix on test set...")
    compute_confusion_matrix(final_model, test_loader, config, is_capsule=True, use_mask_loss=True)

    # Plot training curves
    plot_training_curves({'Full CapFG': history}, save_path='training_curves.png')

    # Background suppression
    print("\nEvaluating background suppression...")
    synth_dataset = SyntheticForegroundDataset(num_samples=1000, size=32, num_classes=10)
    synth_loader = DataLoader(synth_dataset, batch_size=64, shuffle=False)
    iou, leakage = background_suppression_metrics(final_model, synth_loader)
    print(f"Foreground IoU: {iou:.4f} | Background Leakage: {leakage:.4f}")

    # Complexity
    base_model = CapsuleNetAblation(**experiments[0]['model_kwargs']).to(device)
    base_params, base_infer = measure_complexity(base_model, (3,32,32), is_capsule=True, use_mask_loss=False)
    full_params, full_infer = measure_complexity(final_model, (3,32,32), is_capsule=True, use_mask_loss=True)
    print(f"Baseline params: {base_params/1e6:.2f} M, inference: {base_infer:.2f} ms/sample")
    print(f"Full model params: {full_params/1e6:.2f} M, inference: {full_infer:.2f} ms/sample")

    # ResNet-18+CBAM benchmark
    print("\n" + "="*80)
    print("BENCHMARK: ResNet-18 with CBAM")
    print("="*80)
    res_mean, res_std, res_lo, res_hi = multi_run_resnet(resnet18_cbam, {'num_classes':10},
                                                         config, train_loader, val_loader, n_runs=args.n_runs)
    print(f"ResNet+CBAM: {res_mean:.4f} ± {res_std:.4f} | 95% CI [{res_lo:.4f}, {res_hi:.4f}]")
    res_model = resnet18_cbam(num_classes=10).to(device)
    res_params, res_infer = measure_complexity(res_model, (3,32,32), is_capsule=False)
    print(f"ResNet+CBAM params: {res_params/1e6:.2f} M, inference: {res_infer:.2f} ms/sample")

    # Threshold sensitivity
    print("\n" + "="*80)
    print("Threshold sensitivity (0.05–0.20)")
    thresh_df = run_threshold_sensitivity(config, thresholds=(0.05,0.10,0.15,0.20), n_runs=args.n_runs)
    print(thresh_df)

    # Background shift robustness
    print("\n" + "="*80)
    print("Background shift robustness (additive noise)")
    bg_df = evaluate_background_shift(final_model, batch_size=config['batch_size'], noise_levels=(0.1,0.2,0.3))
    print(bg_df)

    # Compile final results table
    final_rows = []
    for name, (mean, std, lo, hi, _) in results.items():
        final_rows.append({
            'Model': name,
            'Mean Val Acc': mean,
            'Std': std,
            '95% CI Lower': lo,
            '95% CI Upper': hi,
            'Params (M)': None,
            'Inference (ms)': None,
            'IoU': None,
            'Leakage': None
        })
    final_rows.append({
        'Model': 'ResNet-18+CBAM',
        'Mean Val Acc': res_mean,
        'Std': res_std,
        '95% CI Lower': res_lo,
        '95% CI Upper': res_hi,
        'Params (M)': res_params/1e6,
        'Inference (ms)': res_infer,
        'IoU': None,
        'Leakage': None
    })
    final_rows.append({
        'Model': 'Full CapFG (final)',
        'Mean Val Acc': best_acc,
        'Std': None,
        '95% CI Lower': None,
        '95% CI Upper': None,
        'Params (M)': full_params/1e6,
        'Inference (ms)': full_infer,
        'IoU': iou,
        'Leakage': leakage
    })
    df_final = pd.DataFrame(final_rows)
    df_final.to_csv('final_complete_results.csv', index=False)
    print("\nSaved final_complete_results.csv")

    # Generate text summary for supervisor
    with open('thesis_result_1_final_summary.txt', 'w') as f:
        f.write("RESULT 1: FOREGROUND-AWARE CAPSULE FEATURE EXTRACTION\n")
        f.write("====================================================\n\n")
        f.write("Ablation study (validation accuracy, 95% CI):\n")
        for name, (mean, std, lo, hi, _) in results.items():
            f.write(f"  {name:20s} {mean:.4f} ± {std:.4f}  [{lo:.4f}, {hi:.4f}]\n")
        f.write(f"\nResNet-18+CBAM benchmark: {res_mean:.4f} ± {res_std:.4f}  [{res_lo:.4f}, {res_hi:.4f}]\n")
        f.write(f"\nFull CapFG model (with mask):\n")
        f.write(f"  Best validation accuracy: {best_acc:.4f}\n")
        f.write(f"  Foreground IoU: {iou:.4f}\n")
        f.write(f"  Background leakage: {leakage:.4f}\n")
        f.write(f"  Parameters: {full_params/1e6:.2f} M\n")
        f.write(f"  Inference time: {full_infer:.2f} ms/sample\n")
        f.write("\nBackground shift robustness (test accuracy under noise):\n")
        bg_df.to_csv(f, index=False)
        f.write("\nThreshold sensitivity (validation accuracy of MaskOnly and Full):\n")
        thresh_df.to_csv(f, index=False)
        f.write("\nAll plots and confusion matrix saved.\n")
    print("Saved thesis_result_1_final_summary.txt")
    print("\n✅ All experiments completed. Final results are ready for the supervisor.")

if __name__ == '__main__':
    main()

Using device: cuda
Ignoring unknown arguments: ['-f', '/root/.local/share/jupyter/runtime/kernel-a4bc77ca-1867-4e5d-8693-a1be956f0d0c.json']

Loading CIFAR-10 (no input mask for main ablation)...


100%|██████████| 170M/170M [00:06<00:00, 26.3MB/s]



ABLATION STUDY (n_runs=3, epochs=30)

--- Baseline ---
  Epoch 01/30 | loss=0.5625 | val_acc=0.4410 | best=0.4410
  Epoch 02/30 | loss=0.4761 | val_acc=0.4606 | best=0.4606
  Epoch 03/30 | loss=0.4424 | val_acc=0.5128 | best=0.5128
  Epoch 04/30 | loss=0.4158 | val_acc=0.5398 | best=0.5398
  Epoch 05/30 | loss=0.3943 | val_acc=0.5692 | best=0.5692
  Epoch 06/30 | loss=0.3802 | val_acc=0.5692 | best=0.5692
  Epoch 07/30 | loss=0.3680 | val_acc=0.5930 | best=0.5930
  Epoch 08/30 | loss=0.3578 | val_acc=0.5856 | best=0.5930
  Epoch 09/30 | loss=0.3498 | val_acc=0.6108 | best=0.6108
  Epoch 10/30 | loss=0.3423 | val_acc=0.6138 | best=0.6138
  Epoch 11/30 | loss=0.3352 | val_acc=0.6242 | best=0.6242
  Epoch 12/30 | loss=0.3309 | val_acc=0.6328 | best=0.6328
  Epoch 13/30 | loss=0.3243 | val_acc=0.6422 | best=0.6422
  Epoch 14/30 | loss=0.3186 | val_acc=0.6468 | best=0.6468
  Epoch 15/30 | loss=0.3153 | val_acc=0.6606 | best=0.6606
  Epoch 16/30 | loss=0.3104 | val_acc=0.6610 | best=0.6610


In [1]:
from scipy import stats

# Per-run validation accuracies from your ablation (extracted from the notebook output)
baseline_accs = [0.6894, 0.6828, 0.7030]   # Run 1,2,3 from Baseline
full_accs     = [0.7172, 0.7238, 0.7246]   # Run 1,2,3 from Full (Mask+CapFG)

t_stat, p_val = stats.ttest_rel(baseline_accs, full_accs)
print(f"Paired t-test between Baseline and Full (Mask+CapFG):")
print(f"  t = {t_stat:.4f}, p = {p_val:.4f}")

Paired t-test between Baseline and Full (Mask+CapFG):
  t = -5.2676, p = 0.0342
